# 01 · Generate owner messages, split by scenario, LOCK
Labels come from the rubric, never the generator. Cost ≈ $2–6 (OpenRouter), ~5 min.

In [ ]:
import sys, json, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from dotenv import load_dotenv; load_dotenv(ROOT / '.env')
import nest_asyncio; nest_asyncio.apply()
import asyncio

In [ ]:
import pt_data as D, pt_pipeline as PL
from pt_llm import LLM
llm = LLM(concurrency=16, cache_name='generate')
text_rows = await PL.generate_text_rows(llm, per_scenario=40)
print(len(text_rows), 'text rows')

In [ ]:
img_train, img_test = await PL.build_image_rows(llm)
print(len(img_train), 'image train rows;', len(img_test), 'image test rows')
print(open(ROOT/'data/generated/image_split_log.json').read())

In [ ]:
text_train, text_test = PL.split_text(text_rows)
PL.write_candidates(text_train, text_test, img_train, img_test)
import pandas as pd
print(len(text_train), 'train /', len(text_test), 'test candidates. Train files are final -> 04_sft can launch now.')
print('Human check: two teammates fill `action` in data/splits/review_test.csv with drop / fix_label:<urgency>')
pd.read_csv(ROOT/'data/splits/review_test.csv').head(40)

In [ ]:
# After review: apply + LOCK (refuses if already locked)
print(PL.apply_review_and_lock())
print(open(ROOT/'data/splits/LOCKED.md').read())